In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round as spark_round

# 1. CARGA DE CONFIGURACIÓN Y CONEXIÓN A MONGO ATLAS
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
COLLECTION_PROCESSED = os.getenv("MONGO_COLLECTION_PROCESSED")

spark = SparkSession.builder \
    .appName("EDA_Ticket_Salida_Jalil") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

print("🔌 Conectando de forma segura a MongoDB Atlas...")
df_limpio = spark.read.format("mongo") \
    .option("uri", MONGO_URI) \
    .option("database", DB_NAME) \
    .option("collection", COLLECTION_PROCESSED) \
    .load()

# 2. INGENIERÍA DE CARACTERÍSTICAS Y CÁLCULO DEL KPI
print("📊 Calculando KPI: Precio promedio por m² por tipología...")

# Creamos la métrica base por registro
df_con_kpi = df_limpio.withColumn("precio_m2", col("precio") / col("m2"))

# Agrupamos por variables estructurales (Tipología) para consolidar las métricas de negocio
df_kpi_agrupado = df_con_kpi.groupBy("dormitorios", "banos") \
    .agg(
        spark_round(avg("precio_m2"), 2).alias("kpi_promedio_precio_m2"),
        spark_round(avg("precio"), 2).alias("precio_promedio"),
        count("precio").alias("volumen_propiedades")
    ) \
    .filter(col("dormitorios").isNotNull() & col("banos").isNotNull()) \
    .orderBy("dormitorios", "banos")

# 3. EXPORTACIÓN PROFESIONAL A LA RAÍZ DEL PROYECTO
print("💾 Convirtiendo a Pandas y guardando dataset para el dashboard...")
pdf_kpi = df_kpi_agrupado.toPandas()

# Se exporta directo a la raíz del workspace para que app.py lo tome sin problemas de rutas
pdf_kpi.to_csv("datos_kpi_real_estate.csv", index=False)
print("✅ ¡Éxito! Archivo 'datos_kpi_real_estate.csv' listo para usarse de inmediato.")

# Apagamos la sesión de Spark de forma limpia
spark.stop()

🔌 Conectando de forma segura a MongoDB Atlas...
📊 Calculando KPI: Precio promedio por m² por tipología...
💾 Convirtiendo a Pandas y guardando dataset para el dashboard...
✅ ¡Éxito! Archivo 'datos_kpi_real_estate.csv' listo para usarse de inmediato.
